# Ensemble Learning: Random Forest vs XGBoost

## Objective

The objective of this notebook is to compare ensemble learning algorithms with a previously developed single machine learning model. The notebook trains and evaluates Logistic Regression, Random Forest, and XGBoost on the Telco Customer Churn dataset. It also compares their performance using accuracy and examines feature importance to understand which variables have the greatest impact on customer churn prediction.

# Import Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# Load Dataset

In [2]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Data Cleaning

In [3]:
df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df.dropna(inplace=True)

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



# Feature Engineering

In [4]:
service_cols = [
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["TotalServices"] = (
    df[service_cols] != "No"
).sum(axis=1)

df["AvgMonthlySpend"] = (
    df["TotalCharges"] /
    (df["tenure"] + 1)
)

# Prepare Features and Target Variable

In [5]:
X = df.drop("Churn", axis=1)

y = (df["Churn"] == "Yes").astype(int)

# Split the Dataset

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Data Preprocessing

In [7]:
numeric_features = X.select_dtypes(
    include=["int64","float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

C:\Users\HP\AppData\Local\Temp\ipykernel_17784\1199262442.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


# Model 1: Logistic Regression

In [8]:
lr_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),

        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

lr_pipeline.fit(X_train, y_train)

lr_pred = lr_pipeline.predict(X_test)

lr_accuracy = accuracy_score(
    y_test,
    lr_pred
)

print(lr_accuracy)

0.7313432835820896


# Model 2: Random Forest

In [9]:
rf_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42
            )
        )
    ]
)

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)

rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

print(rf_accuracy)

0.7903340440653873


# Model 3: XGBoost

In [10]:
xgb_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),

        (
            "classifier",
            XGBClassifier(
                n_estimators=200,
                learning_rate=0.1,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

xgb_pipeline.fit(X_train, y_train)

xgb_pred = xgb_pipeline.predict(X_test)

xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

print(xgb_accuracy)

0.7803837953091685


# Model Comparison

In [11]:
results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy":[
        lr_accuracy,
        rf_accuracy,
        xgb_accuracy
    ]
})

results

,Model,Accuracy
0,Logistic Regression,0.731343
1,Random Forest,0.790334
2,XGBoost,0.780384


# Feature Importance
## Random Forest Feature Importance

In [12]:
feature_names = rf_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = rf_pipeline.named_steps[
    "classifier"
].feature_importances_

importance_df = pd.DataFrame(
    {
        "Feature": feature_names,
        "Importance": importances
    }
).sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(10)

,Feature,Importance
3,num__TotalCharges,0.131489
1,num__tenure,0.113792
5,num__AvgMonthlySpend,0.113285
2,num__MonthlyCharges,0.108067
38,cat__Contract_Month-to-month,0.057719
4,num__TotalServices,0.033668
20,cat__OnlineSecurity_No,0.028948
45,cat__PaymentMethod_Electronic check,0.025501
29,cat__TechSupport_No,0.024717
18,cat__InternetService_Fiber optic,0.023520


# XGBoost Feature Importance

In [13]:
feature_names = xgb_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = xgb_pipeline.named_steps[
    "classifier"
].feature_importances_

importance_df = pd.DataFrame(
    {
        "Feature": feature_names,
        "Importance": importances
    }
).sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(10)

,Feature,Importance
38,cat__Contract_Month-to-month,0.438338
18,cat__InternetService_Fiber optic,0.251500
17,cat__InternetService_DSL,0.038153
20,cat__OnlineSecurity_No,0.024924
40,cat__Contract_Two year,0.019318
29,cat__TechSupport_No,0.018358
39,cat__Contract_One year,0.014463
1,num__tenure,0.011237
37,cat__StreamingMovies_Yes,0.010213
22,cat__OnlineSecurity_Yes,0.009875


## Difference Between Random Forest and XGBoost

Random Forest is an ensemble learning algorithm that builds many decision trees independently using different bootstrap samples of the training data. The final prediction is made by combining the predictions of all trees through majority voting. XGBoost builds decision trees sequentially, where each new tree focuses on correcting the errors made by previous trees using gradient boosting. While Random Forest is simpler and less prone to overfitting, XGBoost often achieves higher accuracy through boosting and regularization but usually requires more parameter tuning.

# Conclusion

In this notebook, Logistic Regression, Random Forest, and XGBoost were trained and evaluated on the Telco Customer Churn dataset. The ensemble models were compared with the baseline Logistic Regression model using accuracy and classification metrics. Feature importance analysis showed which customer attributes had the greatest influence on churn prediction. Overall, ensemble learning provided a more robust approach for this classification task.